In [1]:
import pysheds
from pysheds.grid import Grid
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.colors import ListedColormap, BoundaryNorm
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
import fiona
import xarray as xr
import shapefile
import math
from scipy import stats
import os
from numpy import random
import netCDF4 as nc
from netCDF4 import Dataset
from shapely.geometry import Point, shape, box, mapping, Polygon, MultiPolygon
from shapely.vectorized import contains
from shapely.strtree import STRtree
import matplotlib.path as mpath
from scipy.ndimage import sobel
from rasterio.transform import rowcol
from rasterio.coords import BoundingBox
from rasterio.mask import mask
from rasterio.plot import show
from rasterio.features import shapes

In [2]:
def get_stationdata(station_file):
    # Load data
    df_Q = pd.read_csv(station_file, delimiter=';', encoding='utf-8', skiprows=36)
    
    # Clean column names
    df_Q.columns = df_Q.columns.str.replace(' ', '')
    
    # Convert dates to datetime (keep full daily precision)
    df_Q['YYYY-MM-DD'] = pd.to_datetime(df_Q['YYYY-MM-DD'], format='%Y-%m-%d')
    
    # Crop by date
    start_date = pd.to_datetime('2018-08-31')
    end_date = pd.to_datetime('2025-12-31')
    df_filtered = df_Q[(df_Q['YYYY-MM-DD'] >= start_date) & (df_Q['YYYY-MM-DD'] <= end_date)]
    
    # Extract values
    stream_gauge = df_filtered['Value'].values
    
    # Reset index for dates
    dates = df_filtered['YYYY-MM-DD'].reset_index(drop=True)
    dates = dates.reset_index()
    
    return stream_gauge, dates


In [3]:
def waterpx_count(shp_input, nc_input):

  # Open NetCDF file and extract variables
  with nc.Dataset(nc_input) as dataset:
      watermask = dataset.variables['watermask'][:]
      latitude = dataset.variables['lat'][:]
      longitude = dataset.variables['lon'][:]

  # Load and reproject the shapefile
  shp = gpd.read_file(shp_input).to_crs('EPSG:4326')
  minlon, minlat, maxlon, maxlat = shp.geometry.total_bounds

  # Limit the NetCDF data to the bounding box of the shapefile
  lat_mask = (latitude >= minlat) & (latitude <= maxlat)
  lon_mask = (longitude >= minlon) & (longitude <= maxlon)

  watermask = watermask[lat_mask, :][:, lon_mask]
  lat_filtered = latitude[lat_mask]
  lon_filtered = longitude[lon_mask]

  # Step 1: Create a grid of filtered points
  lon_grid, lat_grid = np.meshgrid(lon_filtered, lat_filtered)
  points = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])

  # Flatten the watermask array to align with points
  watermask_flat = watermask.ravel()

  # Step 2: Load the shapefile and get combined geometry
  shapefile_geom = shp.geometry.unary_union  # Combine all geometries in the shapefile

  # Step 3: Identify points intersecting the shapefile
  intersects_mask = contains(shapefile_geom, points[:, 0], points[:, 1])

  # Step 4: Filter the points and watermask values
  filtered_points = points[intersects_mask]
  filtered_watermask = watermask_flat[intersects_mask]

  # Step 5: Create geometries for intersecting points
  geometries = [Point(lon, lat) for lon, lat in filtered_points]

  # Step 6: Create a GeoDataFrame
  gdf = gpd.GeoDataFrame({'watermask': filtered_watermask}, geometry=geometries, crs='EPSG:4326')


  # markercolormap2= colors.ListedColormap(['white', 'black','blue'])

  # # Assuming gdf and water_percent are defined, and markercolormap2 is valid
  # fig, ax = plt.subplots(1, 1, figsize=(10, 8))
  # gdf.plot(column='watermask', ax=ax, vmin=0, vmax=3, legend=True, markersize=5, cmap=markercolormap2)
  # ax.set_title(nc_input)
  # ax.set_xlabel('Longitude')
  # ax.set_ylabel('Latitude')
  # plt.show()

  # Step 1: Filter the GeoDataFrame where 'watermask' is equal to 2
  filtered_gdf = gdf[gdf['watermask'] == 1]

  water_pixels = len(filtered_gdf)
  total_pixels = len(gdf)

  water_percent = (water_pixels / total_pixels) * 100

  return water_pixels, water_percent, total_pixels


In [4]:
def calc_avg_precip(shp_input, date_input):
    precip_folder = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/PRECIP/RESAMPLED/'
    filename = f"IMERG-Final.CLIM.2001-2022.{date_input}.V07B.nc4"
    
    nc_input = os.path.join(precip_folder, filename)
    
    # Open NetCDF file and extract variables
    with nc.Dataset(nc_input) as dataset:
        precip = dataset.variables['precipitation'][:]  # shape (time, lat, lon) or (lat, lon)
        # print(precip.shape)
        latitude = dataset.variables['lat'][:]  # shape (lat,)
        longitude = dataset.variables['lon'][:]  # shape (lon,)
    
    # Load and reproject the shapefile
    shp = gpd.read_file(shp_input).to_crs('EPSG:4326')
    minlon, minlat, maxlon, maxlat = shp.geometry.total_bounds
    
    # Limit the NetCDF data to the bounding box of the shapefile
    lat_mask = (latitude >= minlat) & (latitude <= maxlat)
    lon_mask = (longitude >= minlon) & (longitude <= maxlon)
    
    # Filter latitude and longitude based on the mask
    lat_filtered = latitude[lat_mask]
    lon_filtered = longitude[lon_mask]
    # print(lat_filtered,lon_filtered)
    
    # Step 1: Create a grid of filtered points
    lon_grid, lat_grid = np.meshgrid(lon_filtered, lat_filtered)
    points = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])
    
    # Check the dimensionality of precip
    if precip.ndim == 2:  # If it's (lat, lon)
        precip_filtered = precip[:,lat_mask][lon_mask,:]  # Apply both lat and lon masks
    elif precip.ndim == 3:  # If it's (time, lat, lon)
        precip_filtered = precip[:, lat_mask, :][:, :, lon_mask]
    
    # Flatten the filtered precipitation array (time dimension included if 3D)
    precip_flat = precip_filtered.ravel()
    
    # Step 2: Load the shapefile and get combined geometry
    shapefile_geom = shp.geometry.unary_union  # Combine all geometries in the shapefile
    
    # Step 3: Identify points intersecting the shapefile
    intersects_mask = contains(shapefile_geom, points[:, 0], points[:, 1])
    
    # Step 4: Filter the points and precipitation values
    filtered_points = points[intersects_mask]
    filtered_precip = precip_flat[intersects_mask]
    
    avg = np.mean(filtered_precip)

    return avg

In [11]:
dates

NameError: name 'dates' is not defined

In [16]:
# Check for debugging
masterlist = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/GRDC_Stations_UPDATED.csv'
# final_path = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/ALL_STATIONS__DAILY_FINAL_2_00.csv'

stations_df = pd.read_csv(masterlist)
# print(stations_df)
station_num = stations_df['grdc_no']
# station_num

final_result = pd.DataFrame()
print(f"Loaded {len(station_num)} stations.") # as of Nov 2025: should be 578 stations total when loading all

import warnings
warnings.filterwarnings("ignore")
failed_stations = [5204252,5204268,5404270]
for s in failed_stations:
    
    station = s
    print("Running station ", station)
    
    data = stations_df[stations_df['grdc_no']==5204252]

    number = station
    region = data['wmo_reg']
    river = data['river']
    name = data['station']
    lat = data['lat']
    lon = data['long']
    area = data['area_delin']
    altitude = data['altitude']
    
    q_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_GRDC_shp/GRDC_shp/{number}_Q_Day.Cmd.txt' # GRDC monthly streamgauge readings as .txt files
    shp_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}/{station}.shp' # all station delineated shapefiles saved in subfolders named by station number
    dem_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}_dem.tif' # cropped .tif files for 15 arc-second DEM saved in subfolders named by station number
    
    results_df = pd.DataFrame(data)
    stream_gauge, dates = get_stationdata(q_file)

    directory = "/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_DAILY_WATERMASK/"
    file_list = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]

    # Calculate water pixel count
    ncdf_list = []
    water_px = np.zeros(len(dates))
    water_pcnt = np.zeros(len(dates)) 
    water_area = np.zeros(len(dates))
    tot_px = np.zeros(len(dates))

    for i in range(0,10):
        date = dates['YYYY-MM-DD'][i]
        date_str = date.strftime('%Y-%m-%d')

        # Query for the correct dir
        #------------
        cutoff = pd.to_datetime('2025-07-27')
        print(date<cutoff)
        if not isinstance(date, pd.Timestamp):
            date = pd.to_datetime(date)
        cutoff = pd.to_datetime('2025-07-27')
        if date < cutoff:
            directory = "/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/CYGNSS_from_PODAAC/Daily/Daily_For_Trend"
        else:
            directory = f"/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_CYGNSS_DAILY_WATERMASK/{date.year}"
        #-------------
        ncdf_name = f'{directory}/cyg.ddmi.{date_str}.l3.uc-berkeley-watermask-daily.a32.d33.nc'
        #ncdf_name = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_DAILY_WATERMASK/cyg.ddmi.{date}.l3.uc-berkeley-watermask-daily.a31.d32.nc'
        ncdf_list.append(ncdf_name)
    for f, filename in enumerate(ncdf_list):
        #print(filename)
        if os.path.isfile(filename):
            print('success')
            pixel_count, pixel_percent, total_pixels = waterpx_count(shp_file, filename)
            print(pixel_count, pixel_percent, total_pixels)
            water_px[f] = pixel_count
            water_pcnt[f] = pixel_percent
            tot_px[f] = total_pixels
            water_area[f] = ((pixel_percent*area)/100)

    precip = np.zeros(len(dates))
    date_inputs = dates['YYYY-MM-DD']
    
    # Dictionary to store monthly averages
    monthly_precip = {}
    
    for index, date in enumerate(date_inputs):
        # Extract the month string 'YYYY-MM'
        month_str = str(date)[5:7]  
    
        # Compute precipitation for this month if not already done
        if month_str not in monthly_precip:
            monthly_precip[month_str] = calc_avg_precip(shp_file, month_str)
    
        # Assign the monthly value to this day
        precip[index] = monthly_precip[month_str]

Loaded 578 stations.
Running station  5204252
True
True
True
True
True
True
True
True
True
True
success
13429 2.4010370105489 559300
success
13449 2.4046129089933843 559300
success
13526 2.4183801180046487 559300
success
13717 2.4525299481494724 559300
success
14309 2.558376542106204 559300
success



KeyboardInterrupt



In [64]:
# Original code
masterlist = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/GRDC_Stations_UPDATED.csv'
# final_path = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/ALL_STATIONS__DAILY_FINAL_2_00.csv'

stations_df = pd.read_csv(masterlist)
# print(stations_df)
station_num = stations_df['grdc_no']
# station_num

final_result = pd.DataFrame()
print(f"Loaded {len(station_num)} stations.") # as of Nov 2025: should be 578 stations total when loading all

import warnings
warnings.filterwarnings("ignore")
failed_stations = [5204252,5204268,5404270]
for s in range(110,112):
    
    station = station_num[s]
    print("Running station ", station)
    
    data = stations_df.iloc[s]

    number = station
    region = data['wmo_reg']
    river = data['river']
    name = data['station']
    lat = data['lat']
    lon = data['long']
    area = data['area_delin']
    altitude = data['altitude']
    
    q_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_GRDC_shp/GRDC_shp/{number}_Q_Day.Cmd.txt' # GRDC monthly streamgauge readings as .txt files
    shp_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}/{station}.shp' # all station delineated shapefiles saved in subfolders named by station number
    dem_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}_dem.tif' # cropped .tif files for 15 arc-second DEM saved in subfolders named by station number
    
    results_df = pd.DataFrame(stations_df.iloc[[s]])
    stream_gauge, dates = get_stationdata(q_file)

    directory = "/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_DAILY_WATERMASK/"
    file_list = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]

    # Calculate water pixel count
    ncdf_list = []
    water_px = np.zeros(len(dates))
    water_pcnt = np.zeros(len(dates)) 
    water_area = np.zeros(len(dates))
    tot_px = np.zeros(len(dates))

    for i in range(0,len(dates)):
        date = dates['YYYY-MM-DD'][i]
        date_str = date.strftime('%Y-%m-%d')

        # Query for the correct dir
        #------------
        print(date<cutoff)
        if not isinstance(date, pd.Timestamp):
            date = pd.to_datetime(date)
        cutoff = pd.to_datetime('2025-07-27')
        if date < cutoff:
            directory = "/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/CYGNSS_from_PODAAC/Daily/Daily_For_Trend"
        else:
            directory = f"/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_CYGNSS_DAILY_WATERMASK/{date.year}"
        #-------------
        ncdf_name = f'{directory}/cyg.ddmi.{date_str}.l3.uc-berkeley-watermask-daily.a32.d33.nc'
        #ncdf_name = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_DAILY_WATERMASK/cyg.ddmi.{date}.l3.uc-berkeley-watermask-daily.a31.d32.nc'
        ncdf_list.append(ncdf_name)
    for f, filename in enumerate(ncdf_list):
        #print(filename)
        if os.path.isfile(filename):
            print('success')
            pixel_count, pixel_percent, total_pixels = waterpx_count(shp_file, filename)
            print(pixel_count, pixel_percent, total_pixels)
            water_px[f] = pixel_count
            water_pcnt[f] = pixel_percent
            tot_px[f] = total_pixels
            water_area[f] = ((pixel_percent*area)/100)

    precip = np.zeros(len(dates))
    date_inputs = dates['YYYY-MM-DD']
    
    # Dictionary to store monthly averages
    monthly_precip = {}
    
    for index, date in enumerate(date_inputs):
        # Extract the month string 'YYYY-MM'
        month_str = str(date)[5:7]  
    
        # Compute precipitation for this month if not already done
        if month_str not in monthly_precip:
            monthly_precip[month_str] = calc_avg_precip(shp_file, month_str)
    
        # Assign the monthly value to this day
        precip[index] = monthly_precip[month_str]

Loaded 578 stations.
Running station  1357600
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


KeyboardInterrupt



In [22]:
masterlist = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/GRDC_Stations_UPDATED.csv'
stations_df = pd.read_csv(masterlist)
station_num = stations_df['grdc_no']
final_result = pd.DataFrame()
print(f"Loaded {len(station_num)} stations.") # as of Nov 2025: should be 578 stations total when loading all

import warnings
warnings.filterwarnings("ignore")
# Loop over selected stations
for s in range(111, 112):

    station = station_num[s]
    print("Running station ", station)

    data = stations_df.iloc[s]
    number = station
    region = data['wmo_reg']
    river = data['river']
    name = data['station']
    lat = data['lat']
    lon = data['long']
    area = data['area_delin']
    altitude = data['altitude']

    # Paths
    q_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_GRDC_shp/GRDC_shp/{number}_Q_Day.Cmd.txt' # GRDC monthly streamgauge readings as .txt files
    shp_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}/{station}.shp'
    dem_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}_dem.tif'

    # Load streamflow and dates
    results_df = pd.DataFrame(stations_df.iloc[[s]])
    stream_gauge, dates = get_stationdata(q_file)  # returns daily timestamps

    # Initialize water pixel arrays
    water_px = np.zeros(len(dates))
    water_pcnt = np.zeros(len(dates))
    water_area = np.zeros(len(dates))
    tot_px = np.zeros(len(dates))

    # Prepare watermask filenames using fast scandir
    ncdf_list = []
    for i in range(len(dates[0:32])):
        date = dates['YYYY-MM-DD'][i]
        if not isinstance(date, pd.Timestamp):
            date = pd.to_datetime(date)

        # Determine directory based on cutoff date
        cutoff = pd.to_datetime('2025-07-27')
        if date < cutoff:
            directory = "/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/CYGNSS_from_PODAAC/Daily/Daily_For_Trend"
        else:
            directory = f"/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_CYGNSS_DAILY_WATERMASK/{date.year}"

        date_str = date.strftime('%Y-%m-%d')
        filename = f"cyg.ddmi.{date_str}.l3.uc-berkeley-watermask-daily.a32.d33.nc"

        # Check if file exists using scandir for speed
        file_exists = any(entry.name == filename and entry.is_file() for entry in os.scandir(directory))
        if file_exists:
            ncdf_list.append(os.path.join(directory, filename))
        else:
            print(f"File not found: {os.path.join(directory, filename)}")

    # Calculate water pixels
    for f, filename in enumerate(ncdf_list):
        pixel_count, pixel_percent, total_pixels = waterpx_count(shp_file, filename)
        water_px[f] = pixel_count
        water_pcnt[f] = pixel_percent
        tot_px[f] = total_pixels
        water_area[f] = (pixel_percent * area) / 100

    # Compute monthly precipitation once per month
    monthly_precip = {}
    dates_pd = pd.to_datetime(dates['YYYY-MM-DD'])
    precip = np.zeros(len(dates))

    for idx, date in enumerate(dates_pd):
        month_str = str(date)[5:7]    # YYYY-MM
        if month_str not in monthly_precip:
            monthly_precip[month_str] = calc_avg_precip(shp_file, month_str)
        precip[idx] = monthly_precip[month_str]

Loaded 578 stations.
Running station  1396102
67 0.767204855147143 8733
74 0.8473605862819191 8733
77 0.8817130424825376 8733
79 0.9046146799496163 8733
80 0.9160654986831558 8733
84 0.9618687736173137 8733
86 0.9847704110843924 8733
85 0.973319592350853 8733
84 0.9618687736173137 8733
83 0.9504179548837742 8733
88 1.0076720485514714 8733
83 0.9504179548837742 8733
75 0.8588114050154586 8733
74 0.8473605862819191 8733
69 0.790106492614222 8733
69 0.790106492614222 8733
76 0.8702622237489981 8733
77 0.8817130424825376 8733
76 0.8702622237489981 8733
76 0.8702622237489981 8733
75 0.8588114050154586 8733
74 0.8473605862819191 8733
76 0.8702622237489981 8733
70 0.8015573113477614 8733
73 0.8359097675483796 8733
72 0.8244589488148402 8733
75 0.8588114050154586 8733
65 0.7443032176800641 8733
66 0.7557540364136035 8733
59 0.6755983052788275 8733
61 0.6984999427459063 8733
61 0.6984999427459063 8733


In [23]:
df_final = pd.DataFrame({'Date': dates['YYYY-MM-DD'], 'Q': stream_gauge,'SWE': water_pcnt, 'SWE_scaled': water_area, 'P': precip})
df_final.to_csv(f"/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/sls_Daily/{number}.csv")
df_final.head()

,Date,Q,SWE,SWE_scaled,P
0,2018-08-31,-999.0,0.767205,76.684348,0.013451
1,2018-09-01,-999.0,0.847361,84.696146,0.191437
2,2018-09-02,-999.0,0.881713,88.129773,0.191437
3,2018-09-03,-999.0,0.904615,90.418858,0.191437
4,2018-09-04,-999.0,0.916065,91.563401,0.191437


In [16]:
number

1396102

In [25]:
import pandas as pd

# Ensure 'Date' is a datetime column
df_final['Date'] = pd.to_datetime(df_final['Date'])

# Filter for September 2018
mask = (df_final['Date'].dt.year == 2018) & (df_final['Date'].dt.month == 9)
df_sep2018 = df_final[mask]

# Compute averages for the columns of interest
avg_Q = df_sep2018['Q'].mean()
avg_SWE = df_sep2018['SWE'].mean()
avg_SWE_scaled = df_sep2018['SWE_scaled'].mean()
avg_P = df_sep2018['P'].mean()

print("Monthly averages for 2018-09:")
print(f"Q: {avg_Q}")
print(f"SWE: {avg_SWE}")
print(f"SWE_scaled: {avg_SWE_scaled}")
print(f"P: {avg_P}")
# 2018-09	-999.0	0.652697	65.238923	0.191437

Monthly averages for 2018-09:
Q: -999.0
SWE: 0.8630100385510898
SWE_scaled: 86.2603536054544
P: 0.19143735688852434


In [69]:
# Delete Files
import os

directory = "/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_DAILY_WATERMASK/"

# Loop over all files in the directory and remove them
for filename in os.listdir(directory):
    file_path = os.path.join(directory, filename)
    if os.path.isfile(file_path):  # Only delete files, not subdirectories
        os.remove(file_path)

print("All files deleted in", directory)


All files deleted in /global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_DAILY_WATERMASK/


In [11]:
import os

dir_save = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/sls_Daily'

# Create the directory (including any missing parent directories)
os.makedirs(dir_save, exist_ok=True)

print(f"Directory is ready: {dir_save}")


Directory is ready: /global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/sls_Daily


# -------------------------------------------
### 12/4/ Test

In [2]:
failed = [2854500, 2853200, 2854100, 2854180, 2854450]
failed

[2854500, 2853200, 2854100, 2854180, 2854450]

In [3]:
# =========================
# IMPORT PACKAGES
# =========================
print('importing packages')
import pysheds
from pysheds.grid import Grid
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.colors import ListedColormap, BoundaryNorm
import geopandas as gpd
import rasterio
import time
import numpy as np
import pandas as pd
import fiona
import xarray as xr
import shapefile
import math
from scipy import stats
import os
from numpy import random
import netCDF4 as nc
from netCDF4 import Dataset
from shapely.geometry import Point, shape, box, mapping, Polygon, MultiPolygon
from shapely.vectorized import contains
from shapely.strtree import STRtree
import matplotlib.path as mpath
from scipy.ndimage import sobel
from rasterio.transform import rowcol
from rasterio.coords import BoundingBox
from rasterio.mask import mask
from rasterio.plot import show
from rasterio.features import shapes
import warnings
warnings.filterwarnings("ignore")

# Parallelization packages
from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed
print('imported packages')

importing packages
imported packages


In [4]:
def get_stationdata(station_file):
    df_Q = pd.read_csv(station_file, delimiter=';', encoding='utf-8', skiprows=36)
    df_Q.columns = df_Q.columns.str.replace(' ', '')
    df_Q['YYYY-MM-DD'] = pd.to_datetime(df_Q['YYYY-MM-DD'], format='%Y-%m-%d')
    start_date = pd.to_datetime('2018-08-31')
    end_date = pd.to_datetime('2025-12-31')
    df_filtered = df_Q[(df_Q['YYYY-MM-DD'] >= start_date) & (df_Q['YYYY-MM-DD'] <= end_date)]
    try:
        stream_gauge = df_filtered['Value'].values
    except: 
        stream_gauge = df_filtered[' Value'].values
    dates = df_filtered['YYYY-MM-DD'].reset_index(drop=True).reset_index()
    return stream_gauge, dates

def waterpx_count(shp_input, nc_input):
    with nc.Dataset(nc_input) as dataset:
        watermask = dataset.variables['watermask'][:]
        latitude = dataset.variables['lat'][:]
        longitude = dataset.variables['lon'][:]

    shp = gpd.read_file(shp_input).to_crs('EPSG:4326')
    minlon, minlat, maxlon, maxlat = shp.geometry.total_bounds

    lat_mask = (latitude >= minlat) & (latitude <= maxlat)
    lon_mask = (longitude >= minlon) & (longitude <= maxlon)
    watermask = watermask[lat_mask, :][:, lon_mask]
    lat_filtered = latitude[lat_mask]
    lon_filtered = longitude[lon_mask]

    lon_grid, lat_grid = np.meshgrid(lon_filtered, lat_filtered)
    points = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])
    watermask_flat = watermask.ravel()
    shapefile_geom = shp.geometry.unary_union
    intersects_mask = contains(shapefile_geom, points[:, 0], points[:, 1])
    filtered_points = points[intersects_mask]
    filtered_watermask = watermask_flat[intersects_mask]
    geometries = [Point(lon, lat) for lon, lat in filtered_points]
    #print('check1')
    gdf = gpd.GeoDataFrame({'watermask': filtered_watermask}, geometry=geometries, crs='EPSG:4326')

    filtered_gdf = gdf[gdf['watermask'] == 1]
    water_pixels = len(filtered_gdf)
    total_pixels = len(gdf)
    water_percent = (water_pixels / total_pixels) * 100
    #print(water_pixels)
    return water_pixels, water_percent, total_pixels

def calc_avg_precip(shp_input, date_input):
    precip_folder = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/PRECIP/RESAMPLED/'
    filename = f"IMERG-Final.CLIM.2001-2022.{date_input}.V07B.nc4"
    nc_input = os.path.join(precip_folder, filename)

    with nc.Dataset(nc_input) as dataset:
        precip = dataset.variables['precipitation'][:]
        latitude = dataset.variables['lat'][:]
        longitude = dataset.variables['lon'][:]

    shp = gpd.read_file(shp_input).to_crs('EPSG:4326')
    minlon, minlat, maxlon, maxlat = shp.geometry.total_bounds
    lat_mask = (latitude >= minlat) & (latitude <= maxlat)
    lon_mask = (longitude >= minlon) & (longitude <= maxlon)
    lat_filtered = latitude[lat_mask]
    lon_filtered = longitude[lon_mask]

    lon_grid, lat_grid = np.meshgrid(lon_filtered, lat_filtered)
    points = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])

    if precip.ndim == 2:
        precip_filtered = precip[:, lat_mask][lon_mask, :]
    elif precip.ndim == 3:
        precip_filtered = precip[:, lat_mask, :][:, :, lon_mask]

    precip_flat = precip_filtered.ravel()
    shapefile_geom = shp.geometry.unary_union
    intersects_mask = contains(shapefile_geom, points[:, 0], points[:, 1])
    filtered_precip = precip_flat[intersects_mask]
    avg = np.mean(filtered_precip)

    return avg

In [36]:
# =========================
# MAIN STATION PROCESSING FUNCTION
# =========================
def process_station(s, station_num, stations_df):
    start_time = time.time() # record time
    print(s)
    station = station_num[s]
    # Exit if station already been run
    '''
    save_path = f"/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/sls_Daily/{station}_parallel.csv"
    if os.path.exists(save_path) and os.path.getsize(save_path) > 0:
        print(f"Skipping station {station} (already processed).")
        return

    save_path2 = f"/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/sls_Daily/{station}_parallel2.csv"
    if os.path.exists(save_path2) and os.path.getsize(save_path2) > 0:
        print(f"Skipping station {station} (already processed).")
        return
    '''
    print(f"Station {station} has not yet been processed. Processing now...")

    data = stations_df.iloc[s]
    number = station
    region = data['wmo_reg']
    river = data['river']
    name = data['station']
    lat = data['lat']
    lon = data['long']
    area = data['area_delin']
    altitude = data['altitude']
    print(f"Processing station {number}")
    q_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_GRDC_shp/GRDC_shp/{number}_Q_Day.Cmd.txt'
    shp_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}/{station}.shp'
    dem_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}_dem.tif'

    # Load streamflow and dates
    stream_gauge, dates = get_stationdata(q_file)
    print(dates)

    water_px = np.zeros(len(dates))
    water_pcnt = np.zeros(len(dates))
    water_area = np.zeros(len(dates))
    tot_px = np.zeros(len(dates))

    # Gather NetCDF files
    ncdf_list = []
    #tempdatelist = dates[-3:]['index']
    for i in range(3): #range(len(dates)):
        print(i)
        date = pd.to_datetime(dates['YYYY-MM-DD'][i])
        print(date)
        cutoff = pd.to_datetime('2025-07-27')
        if date < cutoff:
            directory = "/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/CYGNSS_from_PODAAC/Daily/Daily_For_Trend"
        else:
            directory = f"/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_CYGNSS_DAILY_WATERMASK/{date.year}"

        date_str = date.strftime('%Y-%m-%d')
        filename = f"cyg.ddmi.{date_str}.l3.uc-berkeley-watermask-daily.a32.d33.nc"

        file_exists = any(entry.name == filename and entry.is_file() for entry in os.scandir(directory))
        if file_exists:
            ncdf_list.append(os.path.join(directory, filename))
        else:
            print(f"File not found: {os.path.join(directory, filename)}")

    # Parallel processing for water pixels per NetCDF file
    print('processing waterpx')
    results = Parallel(n_jobs=4)(
        delayed(waterpx_count)(shp_file, f) for f in ncdf_list
    )
    for idx, (pixel_count, pixel_percent, total_pixels) in enumerate(results):
        water_px[idx] = pixel_count
        water_pcnt[idx] = pixel_percent
        tot_px[idx] = total_pixels
        water_area[idx] = (pixel_percent * area) / 100
        print(water_area[idx])

    # Compute monthly precipitation per station
    dates_pd = pd.to_datetime(dates['YYYY-MM-DD'])
    months = sorted(set(str(date)[5:7] for date in dates_pd))
    monthly_precip_values = Parallel(n_jobs=4)(
        delayed(calc_avg_precip)(shp_file, month) for month in months
    )
    monthly_precip = dict(zip(months, monthly_precip_values))
    precip = np.array([monthly_precip[str(date)[5:7]] for date in dates_pd])

    # Save results
    df_final = pd.DataFrame({
        'Date': dates['YYYY-MM-DD'],
        'Q': stream_gauge,
        'SWE': water_pcnt,
        'SWE_scaled': water_area,
        'P': precip
    })
    save_path3 = f"/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/sls_Daily/{station}_test.csv"
    print(f'Saving to {save_path3}')
    df_final.to_csv(save_path3, index=False)
    print(f'Saved')
    end_time = time.time()  # End timer
    elapsed = end_time - start_time
    print(f"Finished running station {number} in {elapsed:.2f} seconds")
    return(dates)


In [67]:
# Inspect station data of failed stations
error_stations = stations_df[stations_df['grdc_no'].isin(failed)]
print(error_stations.index)
print(failed)
soi = 4
station = failed[soi]
number = error_stations.index[soi]
print(station,number)
#----
q_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_GRDC_shp/GRDC_shp/{station}_Q_Day.Cmd.txt'
shp_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}/{station}.shp'
dem_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}_dem.tif'

# Load streamflow and dates
stream_gauge, dates = get_stationdata(q_file)
print(q_file)
dates

Index([127, 128, 129, 131, 132], dtype='int64')
[2854500, 2853200, 2854100, 2854180, 2854450]
2854450 132
/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_GRDC_shp/GRDC_shp/2854450_Q_Day.Cmd.txt


,index,YYYY-MM-DD


In [66]:
station_file = q_file
df_Q = pd.read_csv(station_file, delimiter=';', encoding='utf-8', skiprows=36)
df_Q.columns = df_Q.columns.str.replace(' ', '')
df_Q['YYYY-MM-DD'] = pd.to_datetime(df_Q['YYYY-MM-DD'], format='%Y-%m-%d')
start_date = pd.to_datetime('2018-08-31')
end_date = pd.to_datetime('2025-12-31')
df_filtered = df_Q[(df_Q['YYYY-MM-DD'] >= start_date) & (df_Q['YYYY-MM-DD'] <= end_date)]
try:
    stream_gauge = df_filtered['Value'].values
except: 
    stream_gauge = df_filtered[' Value'].values
dates = df_filtered['YYYY-MM-DD'].reset_index(drop=True).reset_index()
df_Q

ParserError: Error tokenizing data. C error: Expected 1 fields in line 4, saw 2


In [56]:
station = station_num[0]
q_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/sls_GRDC_shp/GRDC_shp/{station}_Q_Day.Cmd.txt'
shp_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}/{station}.shp'
dem_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}_dem.tif'
stream_gauge, dates = get_stationdata(q_file)
dates

AttributeError: module 'pandas' has no attribute 'open_txt'

In [72]:
stations_missed=[#537,
 #539,
 #547,
 #548,
 556,
 557,
 558,
 559,
 560,
 561,
 562,
 563,
 564,
 565,
 566,
 567,
 #568,
 569,
 570,
 571,
 572,
 573,
 574,
 575,
 576,
 577]
stations_missed

[556,
 557,
 558,
 559,
 560,
 561,
 562,
 563,
 564,
 565,
 566,
 567,
 569,
 570,
 571,
 572,
 573,
 574,
 575,
 576,
 577]

In [77]:
#[[537,539,547,548,568]
stations_df.iloc[568]

grdc_no                     5708110
wmo_reg                           5
sub_reg                        5084
river                VICTORIA RIVER
station          COOLIBAH HOMESTEAD
country                          AU
lat                        -15.5515
long                        130.964
area                        44900.0
area_delin              568607.8192
altitude                      22.15
d_start                      1953.0
d_end                        2025.0
d_yrs                          73.0
d_miss                    20.095422
m_start                      1953.0
m_end                        2001.0
m_yrs                          49.0
m_miss                    21.538462
t_start                        1953
t_end                          2025
t_yrs                            73
lta_discharge               160.759
r_volume_yr             5.069695824
r_height_yr             112.9108201
Name: 568, dtype: object

In [33]:
# =========================
# MAIN SCRIPT, no parallel processing
# =========================
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

if __name__ == "__main__":
    print('script starting, with save failed stations logic')
    masterlist = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/GRDC_Stations_UPDATED.csv'
    stations_df = pd.read_csv(masterlist)
    station_num = stations_df['grdc_no']

    print(f"Loaded {len(station_num)} stations.")
    
    failed_stations = []
    for s in [127, 128, 129, 131, 132]:
        print(s)
        dates = process_station(s, station_num, stations_df)


script starting, with save failed stations logic
Loaded 578 stations.
127
127
Station 2853200 has not yet been processed. Processing now...
Processing station 2853200


KeyError: 0

In [27]:
print(error_stations)
station_num

     grdc_no  wmo_reg  sub_reg         river      station country       lat  \
127  2853200        2     2532       NARMADA  GARUDESHWAR      IN  21.88500   
128  2854100        2     2542   BHIMA RIVER       TAKALI      IN  17.40000   
129  2854180        2     2542   BHIMA RIVER      YADGIRI      IN  16.73700   
131  2854450        2     2542  HAGARI RIVER    RAMAPURAM      IN  15.66030   
132  2854500        2     2543  PENNER RIVER      NELLORE      IN  14.47028   

         long     area    area_delin  ...  m_start   m_end  m_yrs     m_miss  \
127  73.65440  89345.0   87663.86599  ...   1949.0  2025.0   77.0   6.681271   
128  75.85000  33916.0   33390.42032  ...   1968.0  2024.0   57.0  43.401760   
129  77.12700  69863.0   69546.26960  ...   1971.0  2024.0   54.0  25.656878   
131  76.96470  23500.0   23444.20644  ...   1971.0  2025.0   55.0  18.683002   
132  79.98889  53290.0  265007.73630  ...   1965.0  2024.0   60.0  41.944444   

     t_start  t_end  t_yrs  lta_discharge  r

0      1159100
1      1159103
2      1159105
3      1159125
4      1159130
        ...   
573    5708148
574    5709100
575    5709105
576    5709110
577    5709112
Name: grdc_no, Length: 578, dtype: int64

In [76]:
stations_df[stations_df['country'] == 'AU']
#132282.715800
#602051.1205
stations_df_sorted = stations_df.sort_values(by="area_delin", ascending=False)
#first10 = stations_df_sorted.index[0:10]
#stations_df_sorted.iloc[first10]
stations_df_sorted.iloc[0:10]

,grdc_no,wmo_reg,sub_reg,river,station,country,lat,long,area,area_delin,...,m_start,m_end,m_yrs,m_miss,t_start,t_end,t_yrs,lta_discharge,r_volume_yr,r_height_yr
164,3265300,3,3651,RIO PARANA,CORRIENTES,AR,-27.9700,-58.8500,1950000.0,2.196625e+06,...,1904.0,2021.0,118.0,0.000000,1904,2021,118,17268.142,544.5681261,279.2657057
216,3625000,3,3251,AMAZON RIVER,ITAPEUA,BR,-4.0578,-63.0278,1769000.0,1.783483e+06,...,NaN,NaN,NaN,NaN,1971,2020,50,87885.833,2771.567629,1566.742583
83,1234150,1,1341,RIVER NIGER,NIAMEY,NE,13.5200,2.0900,700000.0,1.743396e+06,...,1929.0,2012.0,84.0,34.830339,1929,2025,97,879.895,27.74836872,39.64052674
210,3620000,3,3201,AMAZON RIVER,SANTO ANTONIO DO ICA,BR,-3.1017,-67.9356,1134540.0,1.223710e+06,...,NaN,NaN,NaN,NaN,1972,2020,49,57550.959,1814.927043,1599.703001
246,3635030,3,3351,RIO MADERA,MANICORE,BR,-5.8167,-61.3019,1150000.0,1.153169e+06,...,1967.0,1993.0,27.0,11.875000,1967,2020,54,23840.189,751.8242003,653.7601742
247,3635035,3,3351,RIO MADERA,HUMAITA,BR,-7.5028,-63.0183,1090000.0,1.095359e+06,...,NaN,NaN,NaN,NaN,1967,2020,54,22057.054,695.5912549,638.1571146
548,5404271,5,5041,MURRAY RIVER,LOCK 1 DOWNSTREAM,AU,-34.3509,139.6160,955018.0,1.042048e+06,...,NaN,NaN,NaN,NaN,1949,2025,77,182.229,5.746773744,6.017450712
547,5404270,5,5041,MURRAY RIVER,OVERLAND CORNER (417.5 KM),AU,-34.1787,140.2750,1000001.0,1.015677e+06,...,1985.0,2000.0,16.0,0.000000,1985,2025,41,178.936,5.642925696,5.642920053
212,3623100,3,3231,AMAZON RIVER,SAO PAULO DE OLIVENCA,BR,-3.4500,-68.7500,990781.0,1.012589e+06,...,1979.0,1993.0,15.0,0.000000,1973,2020,48,46914.023,1479.480629,1493.246872
539,5204268,5,5041,MURRAY RIVER,LOCK 9 UPSTREAM (764.8 KM),AU,-34.1915,141.5970,991000.0,9.872410e+05,...,1965.0,1984.0,20.0,20.000000,1949,2025,77,178.955,5.64352488,5.694777881


In [ ]:
# =========================
# MAIN SCRIPT
# =========================
if __name__ == "__main__":
    print('script starting, with save failed stations logic')
    masterlist = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/GRDC_Stations_UPDATED.csv'
    stations_df = pd.read_csv(masterlist)
    station_num = stations_df['grdc_no']

    print(f"Loaded {len(station_num)} stations.")
    num_processes = min(cpu_count(), 8)

    timeout_seconds = 7200  # 30 minutes per station
    failed_stations = []
    with Pool(num_processes) as pool:
        results = []
        for s in stations_missed:
            r = pool.apply_async(process_station, (s, station_num, stations_df))
            results.append((s, r))

        for s, r in results:
            try:
                r.get(timeout=timeout_seconds)
            except Exception as e:
                print(f"Station {station_num[s]} timed out or crashed. Skipping. Error: {e}")
                failed_stations.append(station_num[s])
    
                fail_path = "/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/sls_Daily/failed_stations4.txt"
                with open(fail_path, "w") as f:
                    for st in failed_stations:
                        f.write(str(st) + "\n")

                print(f"Saved {len(failed_stations)} failed stations to {fail_path}")


script starting, with save failed stations logic
Loaded 578 stations.
537548547556557558559539







Station 5204252 has not yet been processed. Processing now...Station 5404271 has not yet been processed. Processing now...Station 5607023 has not yet been processed. Processing now...Station 5607025 has not yet been processed. Processing now...Station 5607024 has not yet been processed. Processing now...Station 5607200 has not yet been processed. Processing now...

Station 5204268 has not yet been processed. Processing now...Station 5404270 has not yet been processed. Processing now...



Processing station 5204252

Processing station 5404271Processing station 5607023Processing station 5607024Processing station 5607025Processing station 5607200
Processing station 5404270Processing station 5204268






